[JobSpy Library](https://github.com/speedyapply/JobSpy)
proxy list

In [ ]:
#%pip install -U python-jobspy
#%pip install ipyleaflet # python package for interactive maps in jupyter notebooks
#%pip install xyzservices # for basemaps

import csv
from jobspy import scrape_jobs
from ipyleaflet import Map, Marker, basemaps, MarkerCluster, Popup
from ipywidgets import HTML
import pandas as pd

#import xyzservices.providers as xyz # check if this is needed

In [ ]:
def gen_linkedin_str(excludeTerms,excludeCompanies):
    """
    Generate a job search string with LinkedIn boolean search formatting
    """
    linkedinStr = "\"engineer\""
    indeedStr = "title:engineer"

    if excludeTerms: #add all of the search terms to exlude
        linkedinStr += " NOT ("
        indeedStr += " -title:("
        for term in excludeTerms:
            linkedinStr += f"\"{term}\" OR "
            if len(term.split(" ")) > 1:
                indeedStr += f"\"{term}\" OR "
            else:
                indeedStr += f"{term} OR "
        linkedinStr = linkedinStr[:-4]
        indeedStr = indeedStr[:-4]    
        linkedinStr += ")"
        indeedStr += ")"
        
    if excludeCompanies:
        linkedinStr += " NOT ("
        indeedStr += " -company:("
        for company in excludeCompanies:
            linkedinStr += f"\"{company}\" OR "
            if len(company.split(" ")) > 1:
                indeedStr += f"\"{company}\" OR "
            else:
                indeedStr += f"{company} OR "
        linkedinStr = linkedinStr[:-4]    
        indeedStr = indeedStr[:-4]    
        linkedinStr += ")"
        indeedStr += ")"

    return (linkedinStr, indeedStr)

In [ ]:
# CONTROL PANEL
#IO Controls
saveCSV = False
fileName = "250814_1000jobs.csv"
filePath = "C:/users/broge/Desktop/"

#Job scraper controls
sitesToSearch = ["linkedin","indeed"] #"linkedin", "zip_recruiter", "google", "glassdoor", "bayt", "naukri", "bdjobs"
#searchStr =
locationToSearch = "Seattle, WA"
distanceToSearch = 25
numResults = 200 #number of postings to scrape. Roughly 52s/100jobs
postingAge = 72 #max posting age in hours

#Search String controls
excludeTerms = ["AI","compute", "electrical", "plumbing", "front end", "frontend", "sales", "software", "full stack", "customer support",
                "integration", "civil", "backend", "devops", "machine learning", "geotechnical", "network", "cloud", "aerospace",
                "Customer Service", "rf testing", "data engineer", "avionics", "gpu", "ux engineer", "security", "salesforce",
                "compiler", "business intelligence", "principal", "it application", "transportation engineer", "ai/ml", "propulsion",
                "customer experience", "pcba", "cryptography", "intern","internship", "ML"]
excludeCompanies = ["Anduril","General Dynamics Ordnance and Tactical Systems", "Palantir","Karman Space & Defense","Boeing",
                    "General Dynamics Mission Systems", "AvtechTyee","Rivet Industries"] #"Amazon"
(linkedinStr,indeedStr) = gen_linkedin_str(excludeTerms,excludeCompanies)

#Map controls
center = (47.6,-122.3) #(38,-95) = center of US
zoomLevel = 8 #4 is good to fit whole US
myBasemap = basemaps.OpenTopoMap #This is the type of map eg street, topographical, nighttime, etc. other options:basemaps.OpenStreetMap.Mapnik, basemaps.NASAGIBS.ViirsEarthAtNight2012

In [11]:
print(linkedinStr)
print(indeedStr)

"engineer" NOT ("AI" OR "compute" OR "electrical" OR "plumbing" OR "front end" OR "frontend" OR "sales" OR "software" OR "full stack" OR "customer support" OR "integration" OR "civil" OR "backend" OR "devops" OR "machine learning" OR "geotechnical" OR "network" OR "cloud" OR "aerospace" OR "Customer Service" OR "rf testing" OR "data engineer" OR "avionics" OR "gpu" OR "ux engineer" OR "security" OR "salesforce" OR "compiler" OR "business intelligence" OR "principal" OR "it application" OR "transportation engineer" OR "ai/ml" OR "propulsion" OR "customer experience" OR "pcba" OR "cryptography" OR "intern" OR "internship" OR "ML") NOT ("Anduril" OR "General Dynamics Ordnance and Tactical Systems" OR "Palantir" OR "Karman Space & Defense" OR "Boeing" OR "General Dynamics Mission Systems" OR "AvtechTyee" OR "Rivet Industries")
title:engineer -title:(AI OR compute OR electrical OR plumbing OR "front end" OR frontend OR sales OR software OR "full stack" OR "customer support" OR integration O

[Tips on indeed boolean](https://www.reddit.com/r/jobs/comments/a8b3fj/my_tips_on_how_to_use_boolean_operations_on/)

In [12]:
#initialize empty dataframe
jobs = pd.DataFrame()

jobs = scrape_jobs(
    site_name = "indeed",
    search_term = indeedStr,
    location= locationToSearch,
    distance=distanceToSearch,
    results_wanted= numResults,
    hours_old=postingAge,
    country_indeed="USA"
    #proxies=["5.10.246.207:80","localhost"]
)
print(f"Found {len(jobs)} jobs")

if saveCSV:
    try: 
        jobs.to_csv(f"{filePath}{fileName}", mode='x', quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
        print(f"File '{filePath}{fileName}' created successfully.")
    except FileExistsError:
        print(f"File '{filePath}{fileName}' already exists. Overwriting prevented.")

Found 168 jobs


In [13]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(jobs)

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,in-80bbeeea36c1625f,indeed,https://www.indeed.com/viewjob?jk=80bbeeea36c1625f,https://careers.t-mobile.com/sr-engineer-site-reliability/job/927C4B26E832EB54C9595DA4EF5729FF?source=Indeed,"Sr Engineer, Site Reliability",T-Mobile,"Bellevue, WA, US",2025-10-05,fulltime,direct_data,...,"Bellevue, WA","10,000+",more than $10B (USD),,None,None,None,None,None,None
1,in-c3bd224f29b48a49,indeed,https://www.indeed.com/viewjob?jk=c3bd224f29b48a49,https://jsv3.recruitics.com/redirect?rx_cid=3427&rx_jobId=200618997-3337_rxr-1&rx_url=https%3A%2F%2Fjobs.apple.com%2Fen-us%2Fdetails%2F200618997-3337%2Fsr-ios-engineer-siri-speech%3Fboard_id%3DJB001%26rx_campaign%3Dindeed0%26rx_ch%3Djobp4p%26rx_group%3D130780%26rx_id%3De09eb358-a183-11f0-8bed-1d1b9170bf6c%26rx_job%3D200618997-3337_rxr-1%26rx_medium%3Dcpc%26rx_r%3Dnone%26rx_source%3Dindeed%26rx_ts%3D20251005T103802Z%26rx_vp%3Dcpc%26team%3DMLAI,"Sr. iOS Engineer, Siri Speech",Apple,"Seattle, WA, US",2025-10-04,NaN,direct_data,...,"Cupertino, CA","10,000+",more than $10B (USD),This is where you can do the best work of your life.,None,None,None,None,None,None
2,in-22df897945bc6ab5,indeed,https://www.indeed.com/viewjob?jk=22df897945bc6ab5,http://www.indeed.com/job/structural-engineer-entry-level-experienced-22df897945bc6ab5,Structural Engineer – Entry Level to Experienced,Malsam Tsang Structural Engineering,"Seattle, WA, US",2025-10-04,fulltime,direct_data,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
3,in-f5bff0ffab0f114e,indeed,https://www.indeed.com/viewjob?jk=f5bff0ffab0f114e,https://rr.jobsyn.org/2B379A4D594D4FB287B22554130D57D81554,"Senior Service Engineer IS, Print Administrator - VPSX SME",Providence,"Redmond, WA, US",2025-10-04,fulltime,direct_data,...,"Renton, WA","10,000+",$100M to $500M (USD),"Providence is a not-for-profit network of hospitals, care centers, health plans, clinics, home health care and services continuing a more than 100-year tradition of serving the poor and vulnerable.",None,None,None,None,None,None
4,in-849fddc338ac9946,indeed,https://www.indeed.com/viewjob?jk=849fddc338ac9946,https://www.amazon.jobs/jobs/3101001/sr-hardware-development-engineer--pcie-aws-board-core-design-and-services-team?cmpid=DA_INAD200785B,"Sr. Hardware Development Engineer - PCIe, AWS Board Core Design and Services Team",Amazon Web Services,"Seattle, WA, US",2025-10-03,fulltime,direct_data,...,"Seattle, WA",Decline to state,more than $10B (USD),"Amazon Web Services (AWS) offers the world’s most comprehensive and broadly adopted cloud platform, with over 200 powerful services that enable our customers and our people to make more of an impact.",None,None,None,None,None,None
5,in-80b9339ba36d1c5a,indeed,https://www.indeed.com/viewjob?jk=80b9339ba36d1c5a,https://jsv3.recruitics.com/redirect?rx_cid=3427&rx_jobId=200591627-3337_rxr-1&rx_url=https%3A%2F%2Fjobs.apple.com%2Fen-us%2Fdetails%2F200591627-3337%2Fsite-reliability-engineer-sre-apple-services-engineering-icloud%3Fboard_id%3DJB001%26rx_campaign%3Dindeed0%26rx_ch%3Djobp4p%26rx_group%3D130780%26rx_id%3Dd43fcc19-a0b9-11f0-8458-e152e0fea3c1%26rx_job%3D200591627-3337_rxr-1%26rx_medium%3Dcpc%26rx_r%3Dnone%26rx_source%3Dindeed%26rx_ts%3D20251005T103802Z%26rx_vp%3Dcpc%26team%3DSFTWR,Site Reliability Engineer (SRE) - Apple Services Engineering / iCloud,Apple,"Seattle, WA, US",2025-10-03,NaN,direct_data,...,"Cupertino, CA","10,000+",more than $10B (USD),This is where you can do the best work of your life.,None,None,None,None,None,None
6,in-f3011d69567c36be,indeed,https://www.indeed.com/viewjob?jk=f3011d69567c36be,https://jobs.ourcareerpages.com/job/949670~Primary~14879290?source=Indeed,Project Engineer,Garco Construction Inc.,"Bremerton, WA, US",2025-10-03,NaN,direct_data,...,Spokane,201 to

In [ ]:
def get_coords(cityStateStr):
    """
    This function retrieves the latitude and longitude of a city.
    The job scraper returns a location for each job as a string with the format "city, SS" where SS is a two letter state abbrev.
    The state and city are looked up in a table of geographic data of US cities.
    """
    
    cityStateStr = cityStateStr.replace(", US","") #Indeed results are formatted City, SS, US. Remove the ", US" to make formatting consistent
    [city,state] = cityStateStr.split(", ")

    file_path = r"C:\Users\broge\Documents\GitHub\job-map\uscities.csv"
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if city == row["city"] and state == row["state_id"]:
                return float(row["lat"]), float(row["lng"])
    return ""

#if a jobs dataframe doesn't exist (ie the scraper wasn't just used) assume the user already has a scraped csv
if len(jobs)==0:
    jobs = pd.read_csv(f"{filePath}{fileName}")
    jobs = jobs.fillna('') #if importing a csv, convet any NaN floats to empty strings

#To plot a marker for each job on a map, we want job title, url, company, and location (latitude+longitude)
markerList = []
for i in range(len(jobs.location)):
    #extract relevant info for each entry of jobs dataframe
    locStr = jobs.location[i]
    url = jobs.job_url[i]
    title = jobs.title[i]
    company = jobs.company[i]

    #check if the entry has a location listed; if so then look up the lat,long of the city
    if locStr:
        coords = get_coords(locStr)
        currentMarker = Marker(location=coords,draggable=False)
        currentMessage = HTML()
        currentMessage.value = f"<a href=\"{url}\">{title}</a><br>{company}"
        currentMarker.popup = currentMessage

        markerList.append(currentMarker)

#Generate map with list of markers from above created above
map = Map(basemap=myBasemap, center=center, zoom=zoomLevel)
cluster = MarkerCluster(markers=markerList,max_cluster_radius=1) #default radius = 80 pixels
map.add(cluster)

Map(center=[47.6, -122.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

In [ ]:
print(indeedStr)



title:engineer -title:(compute OR electrical OR plumbing OR "front end" OR frontend OR sales OR software OR "full stack" OR "customer support" OR integration OR civil OR backend OR devops OR "machine learning" OR geotechnical OR network OR cloud OR aerospace) -company:(Anduril)


In [90]:
for item in jobs.title:
    print(item)

Sourcing Engineer II
Sourcing Engineer II
Lead Transportation Engineer / Project Manager – Roads + Highways
Technical Support Engineer
Radar System Engineer (Expert)
Associate or Experienced Quality Engineer
Detection Engineer
Mechanical Hydraulics Design & Analysis Engineer (Associate or Experienced)
Product Review Engineer (Liaison Engineering)
AI/ML - Applied Research Engineer, Machine Translation
Telecom/EV Design Engineer
Mechanical Project Engineer
Mechanical Project Engineer
Principal Customer Experience Engineer
Systems Engineer I
Founding Engineer
Field Engineer - Kitsap WA
Mechanical/Fluids Design Engineer II - Test Development
Propulsion Engineer III - Thruster Design & Development
Principal Customer Experience Engineer
Hardware Engineer
ASIC Design Verification Engineer, Kuiper Modem DV Team
Regional Environmental Engineer, Air Compliance, AWS Environmental Team
Propulsion Flight Operations Engineer, Project Kuiper
Opto-Mechanical Engineer, Reality Labs Research, Hardware E